# 11 - Role Intelligence

**Objective:** Build a clean role master table from occupation_data.csv.

Every role gets a clean ID and name for mapping company-specific job roles to O*NET occupation titles.

In [1]:
import pandas as pd

DATA_PATH = "../data/raw"

occ = pd.read_csv(f"{DATA_PATH}/occupation_data.csv")
print(f"Occupation Data: {occ.shape}")
print(f"Columns: {list(occ.columns)}")
occ.head()

Occupation Data: (1016, 3)
Columns: ['O*NET-SOC Code', 'Title', 'Description']


,O*NET-SOC Code,Title,Description
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, sh..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of ..."
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes..."
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici..."


In [2]:
# O*NET-SOC Code as role ID, Title as role name
print(f"Total occupations: {len(occ)}")
print(f"Unique SOC codes: {occ['O*NET-SOC Code'].nunique()}")
print(f"Unique titles: {occ['Title'].nunique()}")

print("\nSample roles:")
for _, row in occ.head(10).iterrows():
    print(f"  {row['O*NET-SOC Code']}: {row['Title'][:60]}")

Total occupations: 1016
Unique SOC codes: 1016
Unique titles: 1016

Sample roles:
  11-1011.00: Chief Executives
  11-1011.03: Chief Sustainability Officers
  11-1021.00: General and Operations Managers
  11-1031.00: Legislators
  11-2011.00: Advertising and Promotions Managers
  11-2021.00: Marketing Managers
  11-2022.00: Sales Managers
  11-2032.00: Public Relations Managers
  11-2033.00: Fundraising Managers
  11-3012.00: Administrative Services Managers


In [3]:
# Check what the attrition dataset JobRole values are
attrition = pd.read_csv(f"{DATA_PATH}/employee_attrition.csv")
print("Attrition JobRole values:")
print(attrition['JobRole'].value_counts().to_string())

Attrition JobRole values:
JobRole
Sales Executive              326
Research Scientist           292
Laboratory Technician        259
Manufacturing Director       145
Healthcare Representative    131
Manager                      102
Sales Representative          83
Research Director             80
Human Resources               52


In [4]:
# Check for overlaps between attrition JobRole and occupation Title
attrition_roles = set(attrition['JobRole'].unique())
occupation_titles = set(occ['Title'].unique())

direct = attrition_roles & occupation_titles
print(f"Direct matches: {direct if direct else 'NONE'}")

partial = []
for ar in attrition_roles:
    for ot in occupation_titles:
        if ar.lower() in ot.lower() or ot.lower() in ar.lower():
            partial.append((ar, ot))
print(f"\nPartial matches: {len(partial)}")
for ar, ot in partial[:10]:
    print(f"  '{ar}' -> '{ot}'")

Direct matches: NONE

Partial matches: 63
  'Sales Representative' -> 'Solar Sales Representatives and Assessors'
  'Sales Representative' -> 'Sales Representatives, Wholesale and Manufacturing, Except Technical and Scientific Products'
  'Sales Representative' -> 'Sales Representatives, Wholesale and Manufacturing, Technical and Scientific Products'
  'Sales Representative' -> 'Sales Representatives of Services, Except Advertising, Insurance, Financial Services, and Travel'
  'Research Scientist' -> 'Computer and Information Research Scientists'
  'Human Resources' -> 'Human Resources Specialists'
  'Human Resources' -> 'Human Resources Managers'
  'Human Resources' -> 'Human Resources Assistants, Except Payroll and Timekeeping'
  'Manager' -> 'Information Technology Project Managers'
  'Manager' -> 'Farmers, Ranchers, and Other Agricultural Managers'


## Findings

- occupation_data.csv contains 1,016 O*NET occupation titles
- The attrition dataset uses simplified company-specific role names
- There are 0 direct string matches between the two datasets
- Partial matching is possible but imprecise
- A proper mapping table would be needed for production use